# Capa de analisis - EDA y preprocesamiento

Dataset: NASA Exoplanet Archive + Kepler Objects of Interest.

Objetivo de esta capa: entender que hay en los datos crudos, clasificar atributos, revisar calidad de datos y dejar definido el preprocesamiento antes de modelar.

Tipos de datos usados:

- nominal
- binario
- ordinal
- numerico


In [1]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NAVY = "#1C3257"
TERRA = "#AA4B37"
SAND = "#F4EFE6"
INK = "#1A1A1A"
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_FONT = dict(family="Helvetica, Arial, sans-serif", color=INK, size=13)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)


## 1. Funciones auxiliares

Estas funciones siguen el estilo de las actividades anteriores: resumen, clasificacion de atributos, nulos, outliers y limpieza basica de espacios en blanco.


In [2]:
def clasificar_atributos(
    df: pd.DataFrame,
    hints: dict[str, str] | None = None,
    uso: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Clasifica columnas usando solo: nominal, binario, ordinal y numerico."""
    tipos_validos = {"nominal", "binario", "ordinal", "numerico"}
    hints = hints or {}
    uso = uso or {}
    filas = []

    for col in df.columns:
        s = df[col]
        n_unique = int(s.nunique(dropna=True))
        n_missing = int(s.isna().sum())
        pct_missing = round(float(s.isna().mean() * 100), 2)
        dtype = str(s.dtype)

        if col in hints:
            tipo = hints[col]
        elif pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            tipo = "nominal"
        elif n_unique == 2:
            tipo = "binario"
        elif pd.api.types.is_integer_dtype(s) and n_unique <= 12:
            tipo = "ordinal"
        elif pd.api.types.is_numeric_dtype(s):
            tipo = "numerico"
        else:
            tipo = "nominal"

        if tipo not in tipos_validos:
            raise ValueError(f"Tipo no permitido para {col}: {tipo}")

        ejemplos = s.dropna().astype(str).unique()[:4]
        filas.append(
            {
                "columna": col,
                "dtype_pandas": dtype,
                "tipo_dato": tipo,
                "uso": uso.get(col, "analisis"),
                "n_unique": n_unique,
                "n_missing": n_missing,
                "pct_missing": pct_missing,
                "ejemplos": list(ejemplos),
            }
        )

    return pd.DataFrame(filas).sort_values(
        by=["uso", "tipo_dato", "pct_missing", "columna"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)


def limpiar_espacios_blanco(df: pd.DataFrame) -> pd.DataFrame:
    """Quita espacios al inicio/final en columnas nominales y convierte cadenas vacias en NaN."""
    limpio = df.copy()
    columnas_texto = limpio.select_dtypes(include=["object", "string"]).columns
    for col in columnas_texto:
        limpio[col] = limpio[col].astype("string").str.strip()
        limpio[col] = limpio[col].replace({"": pd.NA})
    return limpio


def info_texto(df: pd.DataFrame) -> str:
    buffer = StringIO()
    df.info(buf=buffer)
    return buffer.getvalue()


def resumen_dataframe(nombre: str, df: pd.DataFrame) -> dict:
    return {
        "dataset": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "filas_duplicadas": int(df.duplicated().sum()),
        "celdas_nulas": int(df.isna().sum().sum()),
        "pct_nulos_total": round(float(df.isna().sum().sum() / df.size * 100), 2),
        "columnas_con_nulos": int((df.isna().sum() > 0).sum()),
    }


def columnas_con_mas_nulos(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    tabla = df.isna().agg(["sum", "mean"]).T
    tabla.columns = ["n_missing", "pct_missing"]
    tabla["pct_missing"] = (tabla["pct_missing"] * 100).round(2)
    return tabla.sort_values("pct_missing", ascending=False).head(n)


def resumen_numerico(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    columnas_validas = [col for col in columnas if col in df.columns]
    return df[columnas_validas].describe().T.round(3)


def detectar_outliers_iqr(s: pd.Series, k: float = 1.5) -> pd.Series:
    if not pd.api.types.is_numeric_dtype(s):
        raise ValueError(f"La serie debe ser numerica, recibio {s.dtype}")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return ((s < low) | (s > high)).fillna(False)


def resumen_outliers_iqr(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    filas = []
    for col in columnas:
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        mascara = detectar_outliers_iqr(df[col])
        filas.append(
            {
                "columna": col,
                "outliers": int(mascara.sum()),
                "pct_outliers": round(float(mascara.mean() * 100), 2),
                "min": df[col].min(),
                "mediana": df[col].median(),
                "max": df[col].max(),
            }
        )
    return pd.DataFrame(filas).sort_values("pct_outliers", ascending=False)


def chi_square(df: pd.DataFrame, col_a: str, col_b: str) -> dict:
    sub = df[[col_a, col_b]].dropna()
    observed = pd.crosstab(sub[col_a], sub[col_b])
    chi2, p, dof, expected = stats.chi2_contingency(observed.values)
    expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
    return {
        "chi2": float(chi2),
        "p_value": float(p),
        "dof": int(dof),
        "observed": observed,
        "expected": expected_df,
    }


def distancia_matriz(df: pd.DataFrame, metrica: str = "euclidean") -> np.ndarray:
    no_numericas = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if no_numericas:
        raise ValueError(f"Solo se aceptan columnas numericas. No numericas: {no_numericas}")
    return pairwise_distances(df.values, metric=metrica)


## 2. Carga de datos crudos

Los CSV descargados desde NASA tienen lineas de comentario al inicio. Por eso se usa `comment="#"`.


In [3]:
CANDIDATE_DATA_DIRS = [Path("data"), Path("mineria") / "data"]
DATA_DIR = next(
    (
        data_dir
        for data_dir in CANDIDATE_DATA_DIRS
        if (data_dir / "cumulative_2026.06.01_20.09.17.csv").exists()
        and (data_dir / "PSCompPars_2026.06.01_20.09.10.csv").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError("No se encontraron los CSV. Deben estar en data/ o mineria/data/.")

KEPLER_PATH = DATA_DIR / "cumulative_2026.06.01_20.09.17.csv"
PSCOMP_PATH = DATA_DIR / "PSCompPars_2026.06.01_20.09.10.csv"

kepler_raw = pd.read_csv(KEPLER_PATH, comment="#")
pscomppars_raw = pd.read_csv(PSCOMP_PATH, comment="#")

kepler = limpiar_espacios_blanco(kepler_raw)
pscomppars = limpiar_espacios_blanco(pscomppars_raw)

print(f"Datos cargados desde: {DATA_DIR}")
pd.DataFrame(
    [
        resumen_dataframe("Kepler KOI cumulative", kepler),
        resumen_dataframe("PSCompPars", pscomppars),
    ]
)


Datos cargados desde: data


,dataset,filas,columnas,filas_duplicadas,celdas_nulas,pct_nulos_total,columnas_con_nulos
0,Kepler KOI cumulative,9564,49,0,40104,8.56,36
1,PSCompPars,6291,84,0,78176,14.79,71


In [4]:
print("Kepler KOI")
display(kepler.head(5))

print("PSCompPars")
display(pscomppars.head(5))


Kepler KOI


,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_teq_err1,koi_teq_err2,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_tce_delivname,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,9.488036,2.775000e-05,-2.775000e-05,170.538750,0.002160,-0.002160,0.146,0.318,-0.146,2.95750,0.08190,-0.08190,615.8,19.5,-19.5,2.26,0.26,-0.15,793.0,NaN,NaN,93.59,29.45,-16.65,35.8,1.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,54.418383,2.479000e-04,-2.479000e-04,162.513840,0.003520,-0.003520,0.586,0.059,-0.443,4.50700,0.11600,-0.11600,874.8,35.5,-35.5,2.83,0.32,-0.19,443.0,NaN,NaN,9.11,2.87,-1.62,25.8,2.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,10811496,K00753.01,<NA>,CANDIDATE,CANDIDATE,0.000,0,0,0,0,19.899140,1.494000e-05,-1.494000e-05,175.850252,0.000581,-0.000581,0.969,5.126,-0.077,1.78220,0.03410,-0.03410,10829.0,171.0,-171.0,14.60,3.92,-1.31,638.0,NaN,NaN,39.30,31.04,-10.49,76.3,1.0,q1_q17_dr25_tce,5853.0,158.0,-176.0,4.544,0.044,-0.176,0.868,0.233,-0.078,297.00482,48.134129,15.436
3,10848459,K00754.01,<NA>,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,1.736952,2.630000e-07,-2.630000e-07,170.307565,0.000115,-0.000115,1.276,0.115,-0.092,2.40641,0.00537,-0.00537,8079.2,12.8,-12.8,33.46,8.50,-2.83,1395.0,NaN,NaN,891.96,668.95,-230.35,505.6,1.0,q1_q17_dr25_tce,5805.0,157.0,-174.0,4.564,0.053,-0.168,0.791,0.201,-0.067,285.53461,48.285210,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,2.525592,3.761000e-06,-3.761000e-06,171.595550,0.001130,-0.001130,0.701,0.235,-0.478,1.65450,0.04200,-0.04200,603.3,16.9,-16.9,2.75,0.88,-0.35,1406.0,NaN,NaN,926.16,874.33,-314.24,40.9,1.0,q1_q17_dr25_tce,6031.0,169.0,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509


PSCompPars


,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,pl_controv_flag,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxlim,pl_rade,pl_radeerr1,pl_radeerr2,pl_radelim,pl_radj,pl_radjerr1,pl_radjerr2,pl_radjlim,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_bmasselim,pl_bmassj,pl_bmassjerr1,pl_bmassjerr2,pl_bmassjlim,pl_bmassprov,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccenlim,pl_insol,pl_insolerr1,pl_insolerr2,pl_insollim,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtlim,ttv_flag,st_spectype,st_teff,st_tefferr1,st_tefferr2,st_tefflim,st_rad,st_raderr1,st_raderr2,st_radlim,st_mass,st_masserr1,st_masserr2,st_masslim,st_met,st_meterr1,st_meterr2,st_metlim,st_metratio,st_logg,st_loggerr1,st_loggerr2,st_logglim,rastr,ra,decstr,dec,sy_dist,sy_disterr1,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2
0,11 Com b,11 Com,2,1,Radial Velocity,2007.0,Xinglong Station,0,323.21000,0.06,-0.05,0.0,1.178,0.000,0.000,0.0,12.2,NaN,NaN,0.0,1.09,NaN,NaN,0.0,4914.898486,39.092894,-39.728551,0.0,15.464,0.123,-0.125,0.0,Msini,0.2380,0.0070,-0.0070,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,G8 III,4874.0,NaN,NaN,0.0,13.76,2.85,-2.45,0.0,2.09,0.64,-0.63,0.0,-0.26,0.10,-0.10,0.0,[Fe/H],2.45,0.08,-0.08,0.0,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,93.1846,1.9238,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848
1,11 UMi b,11 UMi,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,0,516.21997,3.20,-3.20,0.0,1.530,0.070,-0.070,0.0,12.3,NaN,NaN,0.0,1.09,NaN,NaN,0.0,4684.814200,794.575000,-794.575000,0.0,14.740,2.500,-2.500,0.0,Msini,0.0800,0.0300,-0.0300,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,K4 III,4213.0,46.0,-46.0,0.0,29.79,2.84,-2.84,0.0,2.78,0.69,-0.69,0.0,-0.02,NaN,NaN,0.0,[Fe/H],1.93,0.07,-0.07,0.0,15h17m05.90s,229.274595,+71d49m26.19s,71.823943,125.3210,1.9765,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903
2,14 And b,14 And,1,1,Radial Velocity,2008.0,Okayama Astrophysical Observatory,0,186.76000,0.11,-0.12,0.0,0.775,0.000,0.000,0.0,13.1,NaN,NaN,0.0,1.16,NaN,NaN,0.0,1131.151301,36.232438,-38.775066,0.0,3.559,0.114,-0.122,0.0,Msini,0.0000,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,K0 III,4888.0,NaN,NaN,0.0,11.55,1.12,-0.51,0.0,1.78,0.43,-0.29,0.0,-0.21,0.10,-0.10,0.0,[Fe/H],2.55,0.06,-0.07,0.0,23h31m17.80s,352.824150,+39d14m09.01s,39.235837,75.4392,0.7140,-0.7140,5.23133,0.023,-0.023,2.331,0.240,-0.240,4.91781,0.002826,-0.002826
3,14 Her b,14 Her,1,2,Radial Velocity,2002.0,W. M. Keck Observatory,0,1766.41000,0.67,-0.68,0.0,2.839,0.039,-0.041,0.0,12.5,NaN,NaN,0.0,1.12,NaN,NaN,0.0,2828.672822,413.176929,-540.308292,0.0,8.900,1.300,-1.700,0.0,Mass,0.3683,0.0029,-0.0029,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,K0,5338.0,25.0,-25.0,0.0,0.93,0.01,-0.01,0.0,0.97,0.04,-0.04,0.0,0.43,0.07,-0.07,0.0,[Fe/H],4.45,0.02,-0.02,0.0,16h10m24.50s,242.602101,+43d48m58.90s,43.816362,17.9323,0.0073,-0.0073,6.61935,0.023,-0.023,4.714,0.016,-0.016,6.38300,0.000351,-0.000351
4,16 Cyg B b,16 Cyg B,3,1,Radial Velocity,1996.0,Multiple Observatories,0,798.50000,1.00,-1.00,0.0,1.660,0.030,-0.030,0.0,13.5,NaN,NaN,0.0,1.20,NaN,NaN,0.0,565.737400,25.426400,-25.426400,0.0,1.780,0.080,-0.080,0.0,Msini,0.6800,0.0200,-0.0200,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,G3 V,5750.0,8.0,-8.0,0.0,1.13,0.01,-0.01,0.0,1.08,0.04,-0.04,0.0,0.06,NaN,NaN,0.0,[Fe/H],4.36,0.01,-0.01,0.0,19h41m51.75s,295.465642,+50d31m00.57s,50.516824,21.1397,0.0110,-0.0111,6.21500,0.016,-0.016,4.651,0.016,-0.016,6.06428,0.000603,-0.000603


## 3. Estructura general: df.info() y df.describe()

`info()` muestra tipos de pandas y nulos. `describe()` resume las variables numericas y ayuda a detectar escalas raras, colas largas y posibles outliers.


In [5]:
print("INFO - Kepler KOI")
print(info_texto(kepler))

print("INFO - PSCompPars")
print(info_texto(pscomppars))


INFO - Kepler KOI
<class 'pandas.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Data columns (total 49 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   kepid              9564 non-null   int64  
 1   kepoi_name         9564 non-null   string 
 2   kepler_name        2747 non-null   string 
 3   koi_disposition    9564 non-null   string 
 4   koi_pdisposition   9564 non-null   string 
 5   koi_score          8054 non-null   float64
 6   koi_fpflag_nt      9564 non-null   int64  
 7   koi_fpflag_ss      9564 non-null   int64  
 8   koi_fpflag_co      9564 non-null   int64  
 9   koi_fpflag_ec      9564 non-null   int64  
 10  koi_period         9564 non-null   float64
 11  koi_period_err1    9110 non-null   float64
 12  koi_period_err2    9110 non-null   float64
 13  koi_time0bk        9564 non-null   float64
 14  koi_time0bk_err1   9110 non-null   float64
 15  koi_time0bk_err2   9110 non-null   float64
 16  koi_impact       

In [6]:
print("DESCRIBE - Kepler KOI")
display(kepler.describe(include="all").T)

print("DESCRIBE - PSCompPars")
display(pscomppars.describe(include="all").T)


DESCRIBE - Kepler KOI


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
kepid,9564.0,NaN,NaN,NaN,7690628.327373,2653459.080974,757450.0,5556034.25,7906892.0,9873066.5,12935144.0
kepoi_name,9564,9564,K00752.01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kepler_name,2747,2747,Kepler-227 b,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
koi_disposition,9564,3,FALSE POSITIVE,4839,NaN,NaN,NaN,NaN,NaN,NaN,NaN
koi_pdisposition,9564,2,FALSE POSITIVE,4847,NaN,NaN,NaN,NaN,NaN,NaN,NaN
koi_score,8054.0,NaN,NaN,NaN,0.480829,0.476928,0.0,0.0,0.334,0.998,1.0
koi_fpflag_nt,9564.0,NaN,NaN,NaN,0.208595,4.76729,0.0,0.0,0.0,0.0,465.0
koi_fpflag_ss,9564.0,NaN,NaN,NaN,0.232748,0.422605,0.0,0.0,0.0,0.0,1.0
koi_fpflag_co,9564.0,NaN,NaN,NaN,0.197512,0.398142,0.0,0.0,0.0,0.0,1.0
koi_fpflag_ec,9564.0,NaN,NaN,NaN,0.120033,0.325018,0.0,0.0,0.0,0.0,1.0


DESCRIBE - PSCompPars


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
pl_name,6291,6291,11 Com b,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hostname,6291,4709,KOI-351,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sy_snum,6291.0,NaN,NaN,NaN,1.101733,0.339946,1.0,1.0,1.0,1.0,4.0
sy_pnum,6291.0,NaN,NaN,NaN,1.762995,1.152975,1.0,1.0,1.0,2.0,8.0
discoverymethod,6291,11,Transit,4651,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
sy_kmagerr1,5968.0,NaN,NaN,NaN,0.040948,0.250464,0.011,0.02,0.023,0.03,9.995
sy_kmagerr2,5957.0,NaN,NaN,NaN,-0.040962,0.256273,-9.995,-0.03,-0.023,-0.02,-0.011
sy_gaiamag,5948.0,NaN,NaN,NaN,12.248129,2.930209,2.36431,10.425075,12.91745,14.660275,20.1861
sy_gaiamagerr1,5948.0,NaN,NaN,NaN,0.000636,0.001782,0.000112,0.000257,0.000364,0.000539,0.063232


## 4. Clasificacion de atributos

Se clasifican las columnas con los cuatro tipos de clase: nominal, binario, ordinal y numerico. La columna `uso` no es un tipo nuevo; solo ayuda a recordar si una columna se conserva, si es objetivo o si se excluye por fuga de datos.


In [7]:
KEPLER_HINTS = {
    "kepid": "nominal",
    "kepoi_name": "nominal",
    "kepler_name": "nominal",
    "koi_disposition": "nominal",
    "koi_pdisposition": "nominal",
    "koi_score": "numerico",
    "koi_fpflag_nt": "binario",
    "koi_fpflag_ss": "binario",
    "koi_fpflag_co": "binario",
    "koi_fpflag_ec": "binario",
    "koi_tce_plnt_num": "ordinal",
}

KEPLER_USO = {
    "kepid": "identificador",
    "kepoi_name": "identificador",
    "kepler_name": "identificador",
    "koi_disposition": "objetivo_clasificacion",
    "koi_prad": "objetivo_regresion",
    "koi_pdisposition": "excluir_fuga",
    "koi_score": "excluir_fuga",
    "koi_fpflag_nt": "excluir_fuga",
    "koi_fpflag_ss": "excluir_fuga",
    "koi_fpflag_co": "excluir_fuga",
    "koi_fpflag_ec": "excluir_fuga",
}

clasificacion_kepler = clasificar_atributos(kepler, hints=KEPLER_HINTS, uso=KEPLER_USO)
clasificacion_kepler


,columna,dtype_pandas,tipo_dato,uso,n_unique,n_missing,pct_missing,ejemplos
0,koi_tce_delivname,string,nominal,analisis,3,346,3.62,"[q1_q17_dr25_tce, q1_q16_tce, q1_q17_dr24_tce]"
1,koi_teq_err1,float64,numerico,analisis,0,9564,100.00,[]
2,koi_teq_err2,float64,numerico,analisis,0,9564,100.00,[]
3,koi_steff_err2,float64,numerico,analisis,376,483,5.05,"[-81.0, -176.0, -174.0, -211.0]"
4,koi_slogg_err1,float64,numerico,analisis,553,468,4.89,"[0.064, 0.044, 0.053, 0.07]"
5,koi_slogg_err2,float64,numerico,analisis,430,468,4.89,"[-0.096, -0.176, -0.168, -0.21]"
6,koi_srad_err1,float64,numerico,analisis,1162,468,4.89,"[0.105, 0.233, 0.201, 0.334]"
7,koi_srad_err2,float64,numerico,analisis,1384,468,4.89,"[-0.061, -0.078, -0.067, -0.133]"
8,koi_steff_err1,float64,numerico,analisis,265,468,4.89,"[81.0, 158.0, 157.0, 169.0]"
9,koi_depth_err1,float64,numerico,analisis,1428,454,4.75,"[19.5, 35.5, 171.0, 12.8]"


In [8]:
PSCOMP_HINTS = {
    "pl_name": "nominal",
    "hostname": "nominal",
    "discoverymethod": "nominal",
    "disc_facility": "nominal",
    "disc_year": "ordinal",
    "pl_controv_flag": "binario",
    "ttv_flag": "binario",
    "sy_snum": "ordinal",
    "sy_pnum": "ordinal",
    "rastr": "nominal",
    "decstr": "nominal",
    "st_spectype": "nominal",
    "pl_bmassprov": "nominal",
    "st_metratio": "nominal",
}

PSCOMP_USO = {
    "pl_name": "identificador",
    "hostname": "identificador",
    "discoverymethod": "analisis",
    "disc_facility": "analisis",
    "disc_year": "analisis",
    "pl_rade": "analisis",
    "pl_orbper": "analisis",
    "st_teff": "analisis",
    "st_rad": "analisis",
    "st_mass": "analisis",
}

clasificacion_pscomppars = clasificar_atributos(pscomppars, hints=PSCOMP_HINTS, uso=PSCOMP_USO)
clasificacion_pscomppars


,columna,dtype_pandas,tipo_dato,uso,n_unique,n_missing,pct_missing,ejemplos
0,pl_eqtlim,float64,binario,analisis,2,1601,25.45,"[0.0, 1.0]"
1,pl_orbsmaxlim,float64,binario,analisis,2,424,6.74,"[0.0, -1.0]"
2,pl_orbperlim,float64,binario,analisis,2,340,5.40,"[0.0, -1.0]"
3,st_logglim,float64,binario,analisis,2,322,5.12,"[0.0, -1.0]"
4,st_tefflim,float64,binario,analisis,2,294,4.67,"[0.0, 1.0]"
...,...,...,...,...,...,...,...,...
79,disc_year,float64,ordinal,analisis,34,1,0.02,"[2007.0, 2009.0, 2008.0, 2002.0]"
80,sy_pnum,int64,ordinal,analisis,8,0,0.00,"[1, 2, 3, 7]"
81,sy_snum,int64,ordinal,analisis,4,0,0.00,"[2, 1, 3, 4]"
82,hostname,string,nominal,identificador,4709,0,0.00,"[11 Com, 11 UMi, 14 And, 14 Her]"


## 5. EDA de datos crudos

Aqui se revisa calidad y estructura: nulos, distribucion de clases, resumen numerico, matriz de correlacion, outliers y relaciones iniciales. Esto todavia no entrena modelos.


In [9]:
print("Kepler - columnas con mas nulos")
display(columnas_con_mas_nulos(kepler, n=12))

print("PSCompPars - columnas con mas nulos")
display(columnas_con_mas_nulos(pscomppars, n=12))


Kepler - columnas con mas nulos


,n_missing,pct_missing
koi_teq_err1,9564.0,100.00
koi_teq_err2,9564.0,100.00
kepler_name,6817.0,71.28
koi_score,1510.0,15.79
koi_steff_err2,483.0,5.05
koi_srad_err1,468.0,4.89
koi_steff_err1,468.0,4.89
koi_slogg_err2,468.0,4.89
koi_slogg_err1,468.0,4.89
koi_srad_err2,468.0,4.89


PSCompPars - columnas con mas nulos


,n_missing,pct_missing
pl_eqterr1,4460.0,70.89
pl_eqterr2,4460.0,70.89
pl_orbeccenerr1,4411.0,70.12
pl_orbeccenerr2,4411.0,70.12
st_spectype,3967.0,63.06
pl_bmasseerr1,3274.0,52.04
pl_bmassjerr2,3274.0,52.04
pl_bmasseerr2,3274.0,52.04
pl_bmassjerr1,3274.0,52.04
pl_insolerr2,2645.0,42.04


In [10]:
distribucion_clases = (
    kepler["koi_disposition"]
    .value_counts(dropna=False)
    .rename_axis("clase")
    .reset_index(name="n")
)
distribucion_clases["pct"] = (distribucion_clases["n"] / distribucion_clases["n"].sum() * 100).round(2)

display(distribucion_clases)

fig = px.bar(
    distribucion_clases,
    x="clase",
    y="n",
    text="pct",
    title="Distribucion de koi_disposition",
    template=PLOTLY_TEMPLATE,
    color="clase",
)
fig.update_layout(font=PLOTLY_FONT, showlegend=False)
fig.show()


,clase,n,pct
0,FALSE POSITIVE,4839,50.6
1,CONFIRMED,2747,28.72
2,CANDIDATE,1978,20.68


In [11]:
print("Metodos de descubrimiento en PSCompPars")
display(
    pscomppars["discoverymethod"]
    .value_counts(dropna=False)
    .head(12)
    .rename_axis("metodo")
    .reset_index(name="n")
)


Metodos de descubrimiento en PSCompPars


,metodo,n
0,Transit,4651
1,Radial Velocity,1181
2,Microlensing,278
3,Imaging,97
4,Transit Timing Variations,41
5,Eclipse Timing Variations,17
6,Orbital Brightness Modulation,9
7,Pulsar Timing,8
8,Astrometry,6
9,Pulsation Timing Variations,2


In [12]:
columnas_numericas_kepler = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag",
]

resumen_numerico(kepler, columnas_numericas_kepler)


,count,mean,std,min,25%,50%,75%,max
koi_period,9564.0,75.671,1334.744,0.242,2.734,9.753,40.715,1.299958e+05
koi_impact,9201.0,0.735,3.349,0.000,0.197,0.537,0.889,1.008060e+02
koi_duration,9564.0,5.622,6.472,0.052,2.438,3.793,6.276,1.385400e+02
koi_depth,9201.0,23791.336,82242.683,0.000,159.900,421.100,1473.400,1.541400e+06
koi_prad,9201.0,102.892,3077.639,0.080,1.400,2.390,14.930,2.003460e+05
koi_teq,9201.0,1085.386,856.351,25.000,539.000,878.000,1379.000,1.466700e+04
koi_insol,9243.0,7745.737,159204.665,0.000,20.150,141.600,870.290,1.094755e+07
koi_model_snr,9201.0,259.895,795.807,0.000,12.000,23.000,78.000,9.054700e+03
koi_steff,9201.0,5706.823,796.858,2661.000,5310.000,5767.000,6112.000,1.589600e+04
koi_slogg,9201.0,4.310,0.433,0.047,4.218,4.438,4.543,5.364000e+00


In [13]:
correlacion_kepler = kepler[columnas_numericas_kepler].corr(numeric_only=True)
correlacion_kepler


,koi_period,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,ra,dec,koi_kepmag
koi_period,1.000000,0.004928,0.037302,-0.009180,0.005135,-0.049097,-0.002603,-0.009614,-0.013552,0.001877,-0.000993,0.005670,0.011479,-0.009858
koi_impact,0.004928,1.000000,0.036955,0.005595,0.677380,-0.009982,-0.003659,-0.000476,0.016070,-0.059275,0.022645,0.022464,-0.001804,-0.009796
koi_duration,0.037302,0.036955,1.000000,0.067275,0.036573,-0.194730,-0.018973,0.083584,0.106203,-0.122239,0.013675,0.030927,-0.028101,-0.098477
koi_depth,-0.009180,0.005595,0.067275,1.000000,0.002558,0.080735,-0.006310,0.579725,0.113608,-0.008365,-0.016826,0.028558,-0.018076,0.042709
koi_prad,0.005135,0.677380,0.036573,0.002558,1.000000,-0.001249,0.002989,-0.001746,-0.013025,-0.097329,0.056669,0.008716,0.003037,-0.022565
koi_teq,-0.049097,-0.009982,-0.194730,0.080735,-0.001249,1.000000,0.422720,0.047546,0.240061,-0.527374,0.439855,0.119479,-0.059060,-0.257340
koi_insol,-0.002603,-0.003659,-0.018973,-0.006310,0.002989,0.422720,1.000000,-0.008197,-0.057205,-0.285928,0.530914,0.026666,-0.013950,-0.069748
koi_model_snr,-0.009614,-0.000476,0.083584,0.579725,-0.001746,0.047546,-0.008197,1.000000,0.142403,-0.045613,-0.010111,0.038810,-0.001677,-0.114525
koi_steff,-0.013552,0.016070,0.106203,0.113608,-0.013025,0.240061,-0.057205,0.142403,1.000000,-0.139534,-0.117195,0.107079,-0.025601,-0.333394
koi_slogg,0.001877,-0.059275,-0.122239,-0.008365,-0.097329,-0.527374,-0.285928,-0.045613,-0.139534,1.000000,-0.639253,-0.081565,0.051178,0.474307


In [14]:
fig = px.imshow(
    correlacion_kepler,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Matriz de correlacion - columnas numericas Kepler",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(font=PLOTLY_FONT)
fig.show()


In [15]:
resumen_outliers_iqr(kepler, columnas_numericas_kepler)


,columna,outliers,pct_outliers,min,mediana,max
3,koi_depth,1798,18.80,0.000000,421.100000,1.541400e+06
7,koi_model_snr,1600,16.73,0.000000,23.000000,9.054700e+03
0,koi_period,1567,16.38,0.241843,9.752831,1.299958e+05
4,koi_prad,1469,15.36,0.080000,2.390000,2.003460e+05
6,koi_insol,1438,15.04,0.000000,141.600000,1.094755e+07
10,koi_srad,985,10.30,0.109000,1.000000,2.299080e+02
2,koi_duration,869,9.09,0.052000,3.792600,1.385400e+02
9,koi_slogg,663,6.93,0.047000,4.438000,5.364000e+00
8,koi_steff,552,5.77,2661.000000,5767.000000,1.589600e+04
5,koi_teq,411,4.30,25.000000,878.000000,1.466700e+04


In [16]:
fig = px.histogram(
    kepler,
    x="koi_prad",
    nbins=80,
    title="Distribucion del radio planetario (koi_prad)",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(font=PLOTLY_FONT)
fig.show()

fig = px.histogram(
    kepler.assign(koi_prad_log=np.log1p(kepler["koi_prad"])),
    x="koi_prad_log",
    nbins=80,
    title="Radio planetario transformado con log1p",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(font=PLOTLY_FONT)
fig.show()


In [17]:
variables_importantes = ["koi_period", "koi_depth", "koi_model_snr", "koi_steff"]

for col in variables_importantes:
    fig = px.histogram(
        kepler,
        x=col,
        color="koi_disposition",
        nbins=60,
        title=f"Distribucion de {col} por clase",
        template=PLOTLY_TEMPLATE,
    )
    fig.update_layout(font=PLOTLY_FONT)
    fig.show()


In [18]:
correlaciones = (
    kepler[columnas_numericas_kepler]
    .corr(numeric_only=True)["koi_prad"]
    .drop("koi_prad")
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .reset_index()
)
correlaciones.columns = ["variable", "correlacion_con_koi_prad"]
correlaciones


,variable,correlacion_con_koi_prad
0,koi_impact,0.677380
1,koi_slogg,-0.097329
2,koi_srad,0.056669
3,koi_duration,0.036573
4,koi_kepmag,-0.022565
5,koi_steff,-0.013025
6,ra,0.008716
7,koi_period,0.005135
8,dec,0.003037
9,koi_insol,0.002989


## 6. Decisiones de preprocesamiento

Con base en el EDA, estas son las decisiones antes de modelar:

1. Quitar espacios en blanco en columnas nominales y convertir cadenas vacias a NaN.
2. Conservar identificadores solo para trazabilidad; no son atributos predictores.
3. Usar `koi_disposition` como objetivo de clasificacion.
4. Usar `koi_prad` como objetivo inicial de regresion y preparar `log1p(koi_prad)` por la cola extrema.
5. Excluir columnas con fuga de datos: `koi_score`, `koi_pdisposition` y `koi_fpflag_*`.
6. Imputar nulos y escalar dentro de un `Pipeline`, ajustado solo con train.
7. Usar one-hot encoding solo si se incluyen variables nominales predictoras. En Kepler se excluyen identificadores y columnas con fuga, por eso el primer modelo queda con variables numericas.


In [19]:
columnas_fuga = [
    "koi_score",
    "koi_pdisposition",
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec",
]

columnas_identificacion = ["kepid", "kepoi_name", "kepler_name"]
objetivo_clasificacion = "koi_disposition"
objetivo_regresion = "koi_prad"

features_numericas = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag",
]

features_nominales = []

pd.DataFrame(
    {
        "grupo": [
            "identificadores_no_predictores",
            "objetivo_clasificacion",
            "objetivo_regresion",
            "features_numericas",
            "features_nominales",
            "excluir_por_fuga",
        ],
        "columnas": [
            ", ".join(columnas_identificacion),
            objetivo_clasificacion,
            objetivo_regresion,
            ", ".join(features_numericas),
            "ninguna por ahora: las nominales disponibles son ids o fuga",
            ", ".join(columnas_fuga),
        ],
    }
)


,grupo,columnas
0,identificadores_no_predictores,"kepid, kepoi_name, kepler_name"
1,objetivo_clasificacion,koi_disposition
2,objetivo_regresion,koi_prad
3,features_numericas,"koi_period, koi_impact, koi_duration, koi_depth, koi_teq, koi_insol, koi_mod..."
4,features_nominales,ninguna por ahora: las nominales disponibles son ids o fuga
5,excluir_por_fuga,"koi_score, koi_pdisposition, koi_fpflag_nt, koi_fpflag_ss, koi_fpflag_co, ko..."


In [20]:
transformadores = [
    (
        "numericas",
        Pipeline(
            steps=[
                ("imputar_mediana", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler()),
            ]
        ),
        features_numericas,
    )
]

if features_nominales:
    transformadores.append(
        (
            "nominales",
            Pipeline(
                steps=[
                    ("imputar_desconocido", SimpleImputer(strategy="constant", fill_value="Desconocido")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            features_nominales,
        )
    )

preprocesador = ColumnTransformer(transformadores, remainder="drop")
preprocesador


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feat

## 7. Separacion train/test antes de ajustar transformaciones

Esta parte es critica: el imputador y el scaler se ajustan con `X_train`, no con todo el dataset. Asi se evita fuga de datos.


In [21]:
kepler_modelo_clasificacion = kepler[
    columnas_identificacion
    + [objetivo_clasificacion, objetivo_regresion]
    + features_numericas
].dropna(subset=[objetivo_clasificacion]).copy()

X_clf = kepler_modelo_clasificacion[features_numericas + features_nominales]
y_clf = kepler_modelo_clasificacion[objetivo_clasificacion]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf,
)

X_train_clf_preparado = preprocesador.fit_transform(X_train_clf)
X_test_clf_preparado = preprocesador.transform(X_test_clf)

pd.DataFrame(
    {
        "particion": ["X_train", "X_test", "y_train", "y_test"],
        "shape": [
            X_train_clf.shape,
            X_test_clf.shape,
            y_train_clf.shape,
            y_test_clf.shape,
        ],
    }
)


,particion,shape
0,X_train,"(7651, 13)"
1,X_test,"(1913, 13)"
2,y_train,"(7651,)"
3,y_test,"(1913,)"


In [22]:
kepler_modelo_regresion = kepler[
    columnas_identificacion
    + [objetivo_regresion]
    + features_numericas
].dropna(subset=[objetivo_regresion]).copy()

X_reg = kepler_modelo_regresion[features_numericas + features_nominales]
y_reg = np.log1p(kepler_modelo_regresion[objetivo_regresion])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42,
)

preprocesador_regresion = ColumnTransformer(transformadores, remainder="drop")
X_train_reg_preparado = preprocesador_regresion.fit_transform(X_train_reg)
X_test_reg_preparado = preprocesador_regresion.transform(X_test_reg)

pd.DataFrame(
    {
        "particion": ["X_train_reg", "X_test_reg", "y_train_reg", "y_test_reg"],
        "shape": [
            X_train_reg.shape,
            X_test_reg.shape,
            y_train_reg.shape,
            y_test_reg.shape,
        ],
    }
)


,particion,shape
0,X_train_reg,"(7360, 13)"
1,X_test_reg,"(1841, 13)"
2,y_train_reg,"(7360,)"
3,y_test_reg,"(1841,)"


In [23]:
pscomppars_preprocesado = pscomppars[
    [
        "pl_name",
        "hostname",
        "discoverymethod",
        "disc_year",
        "disc_facility",
        "pl_orbper",
        "pl_rade",
        "pl_bmasse",
        "pl_eqt",
        "st_teff",
        "st_rad",
        "st_mass",
        "sy_dist",
        "ra",
        "dec",
    ]
].copy()

# Esta tabla se usara principalmente como referencia para dashboard/warehouse, no para entrenar el primer modelo.
for col in ["pl_name", "hostname", "discoverymethod", "disc_facility"]:
    pscomppars_preprocesado[col] = pscomppars_preprocesado[col].fillna("Desconocido")

print("Kepler clasificacion listo:", X_train_clf_preparado.shape, X_test_clf_preparado.shape)
print("Kepler regresion listo:", X_train_reg_preparado.shape, X_test_reg_preparado.shape)
print("PSCompPars referencia:", pscomppars_preprocesado.shape)

display(kepler_modelo_clasificacion.head())
display(pscomppars_preprocesado.head())


Kepler clasificacion listo: (7651, 13) (1913, 13)
Kepler regresion listo: (7360, 13) (1841, 13)
PSCompPars referencia: (6291, 15)


,kepid,kepoi_name,kepler_name,koi_disposition,koi_prad,koi_period,koi_impact,koi_duration,koi_depth,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,ra,dec,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,2.26,9.488036,0.146,2.95750,615.8,793.0,93.59,35.8,5455.0,4.467,0.927,291.93423,48.141651,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,2.83,54.418383,0.586,4.50700,874.8,443.0,9.11,25.8,5455.0,4.467,0.927,291.93423,48.141651,15.347
2,10811496,K00753.01,<NA>,CANDIDATE,14.60,19.899140,0.969,1.78220,10829.0,638.0,39.30,76.3,5853.0,4.544,0.868,297.00482,48.134129,15.436
3,10848459,K00754.01,<NA>,FALSE POSITIVE,33.46,1.736952,1.276,2.40641,8079.2,1395.0,891.96,505.6,5805.0,4.564,0.791,285.53461,48.285210,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,2.75,2.525592,0.701,1.65450,603.3,1406.0,926.16,40.9,6031.0,4.438,1.046,288.75488,48.226200,15.509


,pl_name,hostname,discoverymethod,disc_year,disc_facility,pl_orbper,pl_rade,pl_bmasse,pl_eqt,st_teff,st_rad,st_mass,sy_dist,ra,dec
0,11 Com b,11 Com,Radial Velocity,2007.0,Xinglong Station,323.21000,12.2,4914.898486,NaN,4874.0,13.76,2.09,93.1846,185.178779,17.793252
1,11 UMi b,11 UMi,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,516.21997,12.3,4684.814200,NaN,4213.0,29.79,2.78,125.3210,229.274595,71.823943
2,14 And b,14 And,Radial Velocity,2008.0,Okayama Astrophysical Observatory,186.76000,13.1,1131.151301,NaN,4888.0,11.55,1.78,75.4392,352.824150,39.235837
3,14 Her b,14 Her,Radial Velocity,2002.0,W. M. Keck Observatory,1766.41000,12.5,2828.672822,NaN,5338.0,0.93,0.97,17.9323,242.602101,43.816362
4,16 Cyg B b,16 Cyg B,Radial Velocity,1996.0,Multiple Observatories,798.50000,13.5,565.737400,NaN,5750.0,1.13,1.08,21.1397,295.465642,50.516824


## 8. Salida de la capa de analisis

Esta salida conecta la capa de analisis con las siguientes capas. Se guardan datos ya seleccionados y limpiados en `data/processed/`. No se guardan columnas de fuga como `koi_score`, `koi_pdisposition` o `koi_fpflag_*`.


In [24]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

kepler_procesado = kepler_modelo_clasificacion.copy()
kepler_procesado["koi_prad_log"] = np.log1p(kepler_procesado["koi_prad"])

# Tabla de referencia para planetas confirmados. Se usa en warehouse/dashboard, no como fuga del target Kepler.
pscomppars_procesado = pscomppars_preprocesado.copy()

kepler_processed_path = PROCESSED_DIR / "kepler_koi_processed.csv"
pscomppars_processed_path = PROCESSED_DIR / "pscomppars_processed.csv"

kepler_procesado.to_csv(kepler_processed_path, index=False)
pscomppars_procesado.to_csv(pscomppars_processed_path, index=False)

pd.DataFrame(
    {
        "archivo": [str(kepler_processed_path), str(pscomppars_processed_path)],
        "filas": [len(kepler_procesado), len(pscomppars_procesado)],
        "columnas": [kepler_procesado.shape[1], pscomppars_procesado.shape[1]],
    }
)


,archivo,filas,columnas
0,data\processed\kepler_koi_processed.csv,9564,19
1,data\processed\pscomppars_processed.csv,6291,15
